# Case Study: Análise de Fluxo Financeiro e Fraude no BigQuery
**Cientista de Dados Sênior:** Lucas | **Ambiente:** Google Cloud Platform (BigQuery) | **Dialeto:** GoogleSQL

## 0. Setup e Conexão com o Google Cloud
Neste ambiente, nos conectamos diretamente ao BigQuery para analisar os dados já ingeridos em nuvem. Certifique-se de que o seu ambiente local está autenticado no GCP (ex: `gcloud auth application-default login`).

In [ ]:
# Instalação de dependências caso necessário
# %pip install google-cloud-bigquery db-dtypes plotly --quiet

from google.cloud import bigquery
import pandas as pd
import plotly.express as px

# Inicializa o cliente do BigQuery
# Ele utilizará as credenciais padrão do seu ambiente
client = bigquery.Client(project='dados-492918')

print("✅ Cliente BigQuery conectado com sucesso!")

## 1. Feature Engineering (Criação de View na Nuvem)
Transformamos o `step` em dias e horas diretamente via DDL no BigQuery.

In [ ]:
query_view = """
CREATE OR REPLACE VIEW `dados-492918.dados.vw_transactions_enriched` AS
SELECT
  step,
  type,
  amount,
  nameOrig,
  oldbalanceOrg,
  newbalanceOrig,
  nameDest,
  oldbalanceDest,
  newbalanceDest,
  isFraud,
  isFlaggedFraud,
  CAST(TRUNC((step - 1) / 24) + 1 AS INT64) AS day_of_month,
  MOD((step - 1), 24) AS hour_of_day,
  CASE
    WHEN type IN ('CASH_OUT', 'TRANSFER') THEN 'OUT'
    WHEN type IN ('CASH_IN') THEN 'IN'
    ELSE 'INTERNAL'
  END AS flow_direction
FROM `dados-492918.dados.transactions`;
"""

# Executa a criação da View (sem retorno de dados)
job = client.query(query_view)
job.result()  # Aguarda a conclusão
print("✅ View `vw_transactions_enriched` criada/atualizada com sucesso no BigQuery!")

## 2. PILAR 1: Análise Financeira Geral
**Fluxo de Dinheiro e Entradas vs Saídas.**

In [ ]:
query_pilar1 = """
SELECT
  day_of_month,
  flow_direction,
  SUM(amount) AS volume_financeiro,
  COUNT(*) AS qtd_transacoes
FROM `dados-492918.dados.vw_transactions_enriched`
GROUP BY 1, 2
ORDER BY 1, 2;
"""
df_pilar1 = client.query(query_pilar1).to_dataframe()

# Gráfico Executivo
fig1 = px.area(df_pilar1, x='day_of_month', y='volume_financeiro', color='flow_direction',
               title='Fluxo Financeiro Diário (In/Out/Internal)',
               labels={'volume_financeiro': 'Volume (R$)', 'day_of_month': 'Dia'})
fig1.show()

## 3. PILAR 2: Comportamento de Clientes
**"Baleias" e Frequência.**

In [ ]:
query_pilar2 = """
SELECT
  nameOrig,
  COUNT(*) AS frequencia_transacoes,
  SUM(amount) AS volume_total_movimentado,
  ROUND(AVG(amount), 2) AS ticket_medio
FROM `dados-492918.dados.vw_transactions_enriched`
GROUP BY 1
ORDER BY volume_total_movimentado DESC
LIMIT 10;
"""
df_pilar2 = client.query(query_pilar2).to_dataframe()
print("Top 10 Clientes ('Baleias'):")
display(df_pilar2)

## 4. PILAR 3: Tipos de Transação
**Market Share de Volume.**

In [ ]:
query_pilar3 = """
SELECT
  type,
  COUNT(*) AS qtd_usos,
  SUM(amount) AS volume_financeiro,
  ROUND(
    SAFE_DIVIDE(
      SUM(amount) * 100.0,
      (SELECT SUM(amount) FROM `dados-492918.dados.vw_transactions_enriched`)),
    2)
    AS share_volume_perc
FROM `dados-492918.dados.vw_transactions_enriched`
GROUP BY 1
ORDER BY volume_financeiro DESC;
"""
df_pilar3 = client.query(query_pilar3).to_dataframe()

fig3 = px.pie(df_pilar3, values='volume_financeiro', names='type', hole=0.4,
              title='Market Share de Volume por Canal')
fig3.show()

## 5. PILAR 4: Tempo (Picos de Uso e Sazonalidade)
**Entendendo a carga de infraestrutura e comportamento diário.**

In [ ]:
query_pilar4 = """
SELECT
  hour_of_day,
  COUNT(*) AS total_transacoes,
  ROUND(SUM(amount), 2) AS volume_movimentado
FROM `dados-492918.dados.vw_transactions_enriched`
GROUP BY 1
ORDER BY 1;
"""
df_pilar4 = client.query(query_pilar4).to_dataframe()

fig4 = px.bar(df_pilar4, x='hour_of_day', y='total_transacoes', 
              title='Volume de Transações por Hora do Dia',
              color='volume_movimentado', color_continuous_scale='Viridis')
fig4.show()

## 6. PILAR 5: Inteligência de Fraude
**Canais Críticos e Perdas Associadas.**

In [ ]:
query_pilar5 = """
SELECT
  type,
  SUM(isFraud) AS qtd_fraudes,
  SUM(CASE WHEN isFraud = 1 THEN amount ELSE 0 END) AS valor_em_risco,
  ROUND(AVG(isFraud) * 100, 4) AS taxa_fraude_perc
FROM `dados-492918.dados.vw_transactions_enriched`
GROUP BY 1
HAVING qtd_fraudes > 0
ORDER BY valor_em_risco DESC;
"""
df_pilar5 = client.query(query_pilar5).to_dataframe()
print("Diagnóstico Forensic (ROI de Prevenção):")
display(df_pilar5)